In [ ]:
import sklearnex
sklearnex.patch_sklearn()
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import RobustScaler

In [ ]:
# Filter outliers using IQR
def filter_outliers(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    df[column] = df[column].where((df[column] >= lower_bound) & (df[column] <= upper_bound), None)
    return df

In [ ]:
# Import data and split into train/test sets
from sklearn.model_selection import train_test_split
df = pd.read_csv(Path("e1_nutrients.csv"))
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
# Filter outliers using IQR for each depth and nutrient column, then scale with RobustScaler
train_filtered = train_df.copy()
for depth in train_filtered["Depth"].unique():
    mask = train_filtered["Depth"] == depth
    depth_subset = train_filtered.loc[mask].copy()
    for column in ["NITRATE+NITRITE", "AMMONIA", "SILICATE", "PHOSPHATE"]:
        depth_subset = filter_outliers(depth_subset, column)
    train_filtered.loc[mask] = depth_subset
train_clean = train_filtered.dropna()

scaler = RobustScaler()
y_scaler = RobustScaler()
x_train = scaler.fit_transform(train_clean.drop(columns=["NITRITE"]))
y_train = y_scaler.fit_transform(train_clean[["NITRITE"]]).ravel()
x_test = scaler.transform(test_df.drop(columns=["NITRITE"]))
y_test = y_scaler.transform(test_df[["NITRITE"]]).ravel()

df_filtered = train_filtered
df_scaled = train_clean.copy().astype(float)
cols_to_scale = df_scaled.columns != "NITRITE"
df_scaled.loc[:, cols_to_scale] = x_train

In [ ]:
# Show filtering and scaling results to ensure they are working
plot_columns = list(df.columns[1:])
fig, axes = plt.subplots(ncols=2, nrows=3, figsize=(10, 12), sharex=True)
axes = axes.flatten()

for i, column in enumerate(plot_columns):
    ax = axes[i]
    ax.scatter(train_df['Depth'], train_df[column], color="grey", alpha=0.3, label='Original')
    ax.scatter(df_filtered['Depth']+2, df_filtered[column], color="blue", label='Filtered')
    ax2 = ax.twiny()
    ax2.set_xlim(-1.1, 1.1)
    ax2.scatter(df_scaled['Depth'], df_scaled[column], color="red", alpha=0.5, label='Scaled & Filtered')
    ax2.tick_params(axis='y', labelcolor='tab:orange')
    ax.set_xlabel("Depth")
    ax.set_ylabel(column.capitalize())
    ax.set_title(f"{column.capitalize()} vs Depth")


handles1, labels1 = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
handles = handles1 + handles2
labels = labels1 + labels2
fig.legend(handles, labels, loc="upper center", ncol=3)
axes[-1].axis("off")
plt.tight_layout(rect=[0, 0, 1, 1])
plt.show()

In [ ]:
# try to find best paramaters using grid search
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
mlp = MLPRegressor(max_iter=5000, random_state=42, n_iter_no_change=20)
param_grid = [
    {
        'solver': ['adam'],
        'activation': ['relu', 'tanh'],
        'early_stopping': [True],
        'hidden_layer_sizes': [
            (50, 25),
            (64, 64),
            (80, 80),
            (100, 50, 25),
            (128, 64),
            (100, 100, 100),
        ],
        'alpha': [0.0005, 0.001],
        'learning_rate_init': [0.005, 0.001],
        'validation_fraction': [0.1],
        'n_iter_no_change': [30]
    }
]

grid_search = GridSearchCV(estimator=mlp, param_grid=param_grid, cv=5, n_jobs=8, verbose=2, scoring='neg_mean_squared_error', return_train_score=True)
grid_search.fit(x_train, y_train)

print("Best parameters found: ", grid_search.best_params_)
test_preds = grid_search.best_estimator_.predict(x_test)
test_preds_unscaled = y_scaler.inverse_transform(test_preds.reshape(-1, 1)).ravel()
y_test_unscaled = y_scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()
print("MSE test (scaled):", mean_squared_error(y_test, test_preds))
print("MSE test (unscaled):", mean_squared_error(y_test_unscaled, test_preds_unscaled))
cv_results = grid_search.cv_results_
best_index = grid_search.best_index_
print(f"Best Train Score: {-cv_results['mean_train_score'][best_index]}")
print(f"Best Val Score: {-cv_results['mean_test_score'][best_index]}")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score

regressors = []
regressors.append(("RandomForestRegressor-default", RandomForestRegressor(random_state=42)))
regressors.append(("RF-Restricted", RandomForestRegressor(
    n_estimators=200, 
    max_depth=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42
)))
regressors.append(("LinearRegression-default", LinearRegression()))
regressors.append(("MLPRegressor-default", MLPRegressor(random_state=42, max_iter=2000)))
# params from grid search with best cv mse
regressors.append(("MLPRegressor-tunedadam1", MLPRegressor(n_iter_no_change=30, max_iter=2000, hidden_layer_sizes=(100,50,25), validation_fraction=0.1, learning_rate_init=0.001, alpha=0.0005, early_stopping=True, random_state=42)))

scores = []

for regressor_name, regressor in regressors:
    regressor.fit(x_train, y_train)
    train_preds = regressor.predict(x_train)
    test_preds = regressor.predict(x_test)
    train_r2 = r2_score(y_train, train_preds)
    test_r2 = r2_score(y_test, test_preds)

    test_preds_unscaled = y_scaler.inverse_transform(test_preds.reshape(-1, 1)).ravel()
    y_test_unscaled = y_scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()
    train_preds_unscaled = y_scaler.inverse_transform(train_preds.reshape(-1, 1)).ravel()
    y_train_unscaled = y_scaler.inverse_transform(y_train.reshape(-1, 1)).ravel()

    test_scaled_mse = mean_squared_error(y_test, test_preds)
    test_unscaled_mse = mean_squared_error(y_test_unscaled, test_preds_unscaled)

    train_scaled_mse = mean_squared_error(y_train, train_preds)
    train_unscaled_mse = mean_squared_error(y_train_unscaled, train_preds_unscaled)

    cross_val = cross_val_score(regressor, x_train, y_train, cv=5, scoring='neg_mean_squared_error')

    scores.append((regressor_name, train_r2, test_r2, train_scaled_mse, test_scaled_mse, train_unscaled_mse, test_unscaled_mse, cross_val.mean(), cross_val.std()))

pd.DataFrame(scores, columns=["model", "train_r2", "test_r2", "train_scaled_mse", "test_scaled_mse", "train_unscaled_mse", "test_unscaled_mse", "cross_val_mean", "cross_val_std"]).sort_values("test_unscaled_mse") 

In [ ]:
import numpy as np
feature_names = train_clean.drop(columns=["NITRITE"]).columns

rf_default = regressors[0][1]
importances = rf_default.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10,6))
plt.title("Feature Importance for Predicting Nitrite (Random Forest)")
plt.barh(range(len(indices)), importances[indices], align='center', color='skyblue')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('Relative Importance')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
scores = sorted(scores, key=lambda x: x[6])  # Sort by test_unscaled_mse
regressor_names = [score[0] for score in scores]
r2_values = [score[2] for score in scores]
ms_values = [score[6] for score in scores]

ax1.bar(range(len(regressor_names)), r2_values)
ax1.set_xticks(range(len(regressor_names)))
ax1.set_xticklabels(regressor_names, rotation=45, ha='right')
ax1.set_ylabel('Test R² Score')
ax1.set_title('Test R² Score by Regressor')

ax2.bar(range(len(regressor_names)), ms_values)
ax2.set_xticks(range(len(regressor_names)))
ax2.set_xticklabels(regressor_names, rotation=45, ha='right')
ax2.set_ylabel('Test Mean Squared Error')
ax2.set_title('Test Mean Squared Error by Regressor')

plt.tight_layout()
plt.show()